# RSNA EXP-007 labels005 + S+C+A fold0

唯一变量: planes `axial` → `sagittal,coronal,axial`（相对 BEST EXP-005）
标签: **EXP-005 plumbing 固定**（不用 EXP-006 词表，0.680 已证伪）
训练超参: epochs=3 batch=8 n-clips=2 only-fold=0 seed=42 与 004/005/006 一致
对照: EXP-005 LB=**0.699** / EXP-004 Axial=0.687 / EXP-003 S+C=0.685
- rsna_v3 `1612f21b81b0abb56735a5675504043aced36a468c25328004d7bf35f5f02630`
- labels `15cf06b5f8ef9cd3e0e2d27727274c4fa44f8bb6d7c2ee695cfa5aaad3514c19`
- 依据: 平面多样性高（Axial–Sag corr 0.34）；多平面 fold0 已优于堆 fold
- 预计 GPU: 三平面顺序训，以 meta.json 耗时为准（参考单平面 2–3.5h）
- 提交时 eval 会在隐藏 test 上重跑；不自动提交


In [ ]:
# 提交模式:禁网(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 依赖全部离线安装
# 坑1:Kaggle 挂载把文件名中 '+' 去掉(2.2.0cu118=非法 PEP440)→ 复制为合法名
# 坑2:torch 2.2.0+cu118 的 CUDA 库在 nvidia-*-cu11 依赖包中(不在 torch wheel 内),
#     只装 torch 会缺 libcudart(v8/v9 实证)→ 必须显式安装全部 13 个 wheel
# 坑3:/kaggle/working 会被收集为 kernel 输出 → 装完立即删除
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('挂载 wheel 共', len(wh), '个')
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 离线 wheel 不足(应 13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('安装 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 显式安装全部 wheel(--no-deps --no-index: 无版本解析/无联网)
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# preflight: competition mount + all-plane coverage
# layout: series_dir/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
import os, csv
from collections import defaultdict

base = '/kaggle/input'

def _find(b, depth=0, max_depth=3):
    if not os.path.isdir(b):
        return None
    try:
        entries = sorted(os.listdir(b))
    except OSError:
        return None
    for e in entries:
        p = os.path.join(b, e)
        if not os.path.isdir(p):
            continue
        if e in ('train_series', 'test_series'):
            continue
        if os.path.isfile(os.path.join(p, 'test.csv')) and os.path.isdir(os.path.join(p, 'test_series')):
            return p
        if depth < max_depth - 1:
            r = _find(p, depth + 1)
            if r:
                return r
    return None

root = _find(base)
if root is None:
    raise SystemExit('FAIL: 未找到含 test_series+test.csv 的比赛挂载')
print('root =', root)

def count_dcm(dirpath):
    if not os.path.isdir(dirpath):
        return -1
    return sum(1 for f in os.listdir(dirpath) if f.lower().endswith('.dcm'))

def plane_stats(series_csv, series_dir, label):
    if not os.path.isfile(series_csv):
        print(label, 'series csv missing', series_csv)
        return
    with open(series_csv, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise SystemExit(f'FAIL: {label} series csv empty')
    plane_key = None
    for k in rows[0].keys():
        if 'plane' in k.lower() or 'anatom' in k.lower():
            plane_key = k
            break
    uid_key = 'StudyInstanceUID'
    ser_key = 'SeriesInstanceUID'
    study_sets = defaultdict(set)
    plane_series = defaultdict(list)
    for r in rows:
        pl = r.get(plane_key, '?') if plane_key else '?'
        if uid_key in r:
            study_sets[pl].add(r[uid_key])
        plane_series[pl].append(r)
    print(label, 'plane study counts', {k: len(v) for k, v in study_sets.items()})
    for pl in ['Axial', 'Sagittal', 'Coronal']:
        assert len(study_sets.get(pl, set())) > 0, f'FAIL: {label} 无 {pl} study'
        print(label, pl, 'studies', len(study_sets[pl]), 'series', len(plane_series[pl]))
    # sample one dcm dir per plane
    for pl in ['Axial', 'Sagittal', 'Coronal']:
        hit = 0
        for r in plane_series[pl][:4]:
            suid, seid = r[uid_key], r[ser_key]
            for c in [os.path.join(series_dir, suid, seid), os.path.join(series_dir, seid)]:
                k = count_dcm(c)
                if k and k > 0:
                    hit += 1
                    print(f'  {pl} dcm OK study={suid[:12]} series={seid[:14]} n={k}')
                    break
        print(label, pl, 'dcm hits in sample', hit)
        if label == 'test':
            assert hit > 0, f'FAIL: test {pl} DICOM sample empty'

plane_stats(os.path.join(root, 'test_series.csv'), os.path.join(root, 'test_series'), 'test')
plane_stats(os.path.join(root, 'train_series.csv'), os.path.join(root, 'train_series'), 'train')
print('preflight all-planes: OK')


In [ ]:
import base64, os, hashlib
parts=[]; parts.append("...")
path="/kaggle/working/rsna_v3.py"
open(path,"w",encoding="utf-8").write(base64.b64decode(parts[0]).decode("utf-8"))
h=hashlib.sha256(open(path,"rb").read()).hexdigest()
print("rsna_v3",h); assert h=="1612f21b81b0abb56735a5675504043aced36a468c25328004d7bf35f5f02630"; assert "_fold_dir" in open(path,encoding="utf-8").read()
print("v3 OK")


In [ ]:
import base64, os, hashlib
parts=[]; parts.append("...")
path="/kaggle/working/rsna_report_labels.py"
open(path,"w",encoding="utf-8").write(base64.b64decode(parts[0]).decode("utf-8"))
h=hashlib.sha256(open(path,"rb").read()).hexdigest()
print("labels005",h); assert h=="15cf06b5f8ef9cd3e0e2d27727274c4fa44f8bb6d7c2ee695cfa5aaad3514c19"
assert "__soft" in open(path,encoding="utf-8").read()
print("labels005 OK")


In [ ]:
import subprocess, sys, os, json
os.chdir("/kaggle/working")
cmd=[sys.executable,"-u","rsna_v3.py","--planes","sagittal,coronal,axial",
     "--epochs","3","--folds","5","--only-fold","0","--batch","8","--n-clips","2",
     "--experiment-id","exp_007_l005_sca_fold0","--fuse","mean","--verbose"]
print("=== EXP-007 labels005 + S+C+A fold0 ===", flush=True)
print(" ".join(cmd), flush=True)
r=subprocess.run(cmd)
print("returncode", r.returncode, flush=True)
import numpy as np, pandas as pd
root="/kaggle/working/artifacts/exp_007_l005_sca_fold0"
canon=None
for p in [os.path.join(root,"test_uid.csv"), os.path.join(root,"fold0","Axial","test_uid.csv")]:
    if os.path.isfile(p):
        canon=pd.read_csv(p)["StudyInstanceUID"].tolist(); print("uids",len(canon),p); break
for plane in ["Sagittal","Coronal","Axial"]:
    d=os.path.join(root,"fold0",plane)
    ok=all(os.path.isfile(os.path.join(d,n)) for n in ["test_pred.npy","test_uid.csv","meta.json"])
    print(f"[artifact] fold0/{plane}: {{'PASS' if ok else 'MISSING'}}")
    if ok:
        meta=json.load(open(os.path.join(d,"meta.json"),encoding="utf-8"))
        pred=np.load(os.path.join(d,"test_pred.npy"))
        print(" ", plane, "train_s", meta.get("fold_train_time_sec"), "status", meta.get("completion_status"),
              "shape", pred.shape, "mean", float(pred.mean()))
        assert meta.get("completion_status")=="done"
if os.path.isfile("/kaggle/working/submission.csv"):
    s=pd.read_csv("/kaggle/working/submission.csv")
    print("[sub]", s.shape, "NaN", int(s.isna().sum().sum()), "uid_match",
          s["StudyInstanceUID"].tolist()==canon if canon else "n/a")
    print(s.head(3).to_string())
print("[exp007] done")
